# Jolia × FiftyOne — Exploring a 3D CT Vision-Language Foundation Model

Interactive [FiftyOne](https://voxel51.com) exploration of **[Jolia](https://huggingface.co/raidium/Jolia)**,
Raidium's 3D CT vision-language foundation model
([paper](https://arxiv.org/abs/2606.24570), [blog](https://raidium.eu/blog/jolia-foundation-model/)),
on **real chest CT** from public
**[CT-RATE](https://huggingface.co/datasets/ibrahimhamamci/CT-RATE)**.

Jolia emits a global `[CLS]` embedding **and 102 named per-organ embeddings** via learnable "concept
queries" — no segmentation masks. Each demo is a named FiftyOne **saved view**:

| Saved view | What it shows |
|---|---|
| `1 · Embedding Atlas` | UMAP of Jolia's global `[CLS]` embedding, ready to color by finding |
| `2 · Per-Organ Similarity (lungs)` | a lung-finding probe + its **lung**-nearest neighbors |
| `3 · Organ Contrast (lungs vs liver)` | the *same* probe sorted by liver — watch the neighbors scatter |
| `4 · Attention Overlays` | organ cross-attention heatmaps (clean lungs + tricky pancreas) |
| `5 · Probe Errors (FP+FN)` | linear-probe disagreements with the CT-RATE label |

> **Why lungs, not liver?** CT-RATE is a **chest** dataset — every scan is a thorax, and the liver is barely
> in the field of view. So the *hero* per-organ demo uses **lungs** (well-represented, and visible in the
> thumbnail), and keeps **liver** as a deliberate *contrast*: switching to an organ that's barely in frame
> makes the neighbor set scatter, which is exactly the point. The true liver demo belongs on **abdominal**
> CT — see the Merlin-Abd-CT note in §7.

> ⚠️ **Research preview, not a medical device.** Jolia is a research feature extractor; it does not diagnose.
> CT-RATE labels are model-extracted from reports — not clinical ground truth.

---

### Before you start

**1. Create and activate a virtual environment**, then install FiftyOne:

```bash
python3 -m venv .venv
source .venv/bin/activate          # macOS / Linux
# .venv\Scripts\activate           # Windows (PowerShell)
pip install "fiftyone>=1.18"
jupyter lab                        # or: jupyter notebook
```

**2. Get access to CT-RATE** (a gated dataset): create a free [Hugging Face](https://huggingface.co)
account, accept the terms at
[huggingface.co/datasets/ibrahimhamamci/CT-RATE](https://huggingface.co/datasets/ibrahimhamamci/CT-RATE),
then authenticate so the download cell can reach it:

```bash
huggingface-cli login              # or set the HF_TOKEN environment variable
```

**Hardware:** Jolia is only ~22M parameters, so it runs comfortably on CPU or Apple-Silicon **MPS** — the
notebook auto-detects (`mps` → `cuda` → `cpu`). No GPU required. The default subset (`NUM_VOLUMES = 40`)
finishes on a laptop in a few minutes plus download time.

**Idempotent:** re-running skips existing downloads, embeddings, thumbnails, brain runs, and saved views.
All artifacts are written under `./data/` relative to the notebook. Set `FORCE_RECOMPUTE = True` in the
config cell to override.

## 0 · Configuration & dependencies

In [ ]:
%pip install -q \
    torch torchvision \
    timm einops safetensors \
    nibabel \
    huggingface_hub \
    umap-learn \
    scikit-learn \
    pillow tqdm matplotlib

import fiftyone as fo
print("FiftyOne:", fo.__version__)

In [ ]:
from pathlib import Path
import json, os

# ----------------------------- knobs -----------------------------
NUM_VOLUMES     = 40
DATASET_NAME    = "ct_rate_jolia_demo"
DATA_ROOT       = Path("./data/ct_rate").resolve()
THUMB_ROOT      = Path("./data/ct_thumbs").resolve()
OVERLAY_ROOT    = Path("./data/ct_overlays").resolve()
CACHE_ROOT      = Path("./data/cache").resolve()
JOLIA_REV       = "261abf87b0b1d77e74a75ecf0d6e0ca04043fb01"
DEVICE          = None
FORCE_RECOMPUTE = False

FOCUS_ORGANS    = ["lungs", "liver", "kidneys", "heart", "spleen", "pancreas"]

# --- per-organ similarity demo (chest data => lungs is the hero) ---
HERO_ORGAN      = "lungs"      # well-represented in chest CT + visible in the thumbnail
CONTRAST_ORGAN  = "liver"      # barely in frame on chest scans => neighbors scatter (the teaching moment)
PROBE_FINDING   = "Lung nodule"  # pick a probe that HAS this finding, so neighbors visibly share pathology

# --- attention overlays (small: occlusion fallback is expensive) ---
ATTN_ORGANS     = ["lungs", "pancreas"]   # clean (hero) + deliberately-tricky (paper Fig. 9)
ATTN_SAMPLES    = 3
ATTN_GRID       = 6
# -----------------------------------------------------------------

for p in (DATA_ROOT, THUMB_ROOT, OVERLAY_ROOT, CACHE_ROOT):
    p.mkdir(parents=True, exist_ok=True)

import torch
if DEVICE is None:
    DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| force_recompute:", FORCE_RECOMPUTE, "| hero organ:", HERO_ORGAN)

## 1 · Download a CT-RATE subset (idempotent)

Skips the slow repo listing if the volumes are already on disk; `hf_hub_download` caches regardless.
A 401 means you haven't accepted the dataset terms / logged in.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
import pandas as pd
from tqdm.auto import tqdm

REPO_ID = "ibrahimhamamci/CT-RATE"
api = HfApi()

label_csv = hf_hub_download(
    REPO_ID, filename="dataset/multi_abnormality_labels/valid_predicted_labels.csv",
    repo_type="dataset", local_dir=str(DATA_ROOT),
)
labels_df = pd.read_csv(label_csv)
finding_cols = [c for c in labels_df.columns if c != "VolumeName"]
label_lookup = labels_df.set_index("VolumeName")[finding_cols].to_dict("index")
print("Label matrix:", labels_df.shape)
print("Available findings:", finding_cols)

reports = {}
try:
    report_csv = hf_hub_download(
        REPO_ID, filename="dataset/radiology_text_reports/validation_reports.csv",
        repo_type="dataset", local_dir=str(DATA_ROOT),
    )
    rdf = pd.read_csv(report_csv)
    tcol = next((c for c in rdf.columns if c.lower() in ("findings","findings_en","impressions","report")), None)
    kcol = next((c for c in rdf.columns if "volume" in c.lower() or "name" in c.lower()), None)
    if tcol and kcol:
        reports = dict(zip(rdf[kcol].astype(str), rdf[tcol].astype(str)))
    print("Reports loaded:", len(reports))
except Exception as e:
    print("Reports CSV not fetched (non-fatal):", e)

In [ ]:
existing = sorted(DATA_ROOT.glob("dataset/valid_fixed/**/*.nii.gz"))
if not FORCE_RECOMPUTE and len(existing) >= NUM_VOLUMES:
    local_vols = existing[:NUM_VOLUMES]
    print(f"Found {len(existing)} volumes on disk; using {len(local_vols)} (no download).")
else:
    print("Listing repo files (one-time, ~30s)...")
    all_files = api.list_repo_files(REPO_ID, repo_type="dataset", revision="main")
    vol_files = sorted(f for f in all_files
                       if f.startswith("dataset/valid_fixed/") and f.endswith(".nii.gz"))[:NUM_VOLUMES]
    local_vols = []
    for f in tqdm(vol_files, desc="CT volumes"):
        p = hf_hub_download(REPO_ID, filename=f, repo_type="dataset", local_dir=str(DATA_ROOT))
        local_vols.append(Path(p))

def volume_name(path): return path.name  # label keys include the .nii.gz suffix

manifest = []
for p in local_vols:
    vname = volume_name(p)
    row = label_lookup.get(vname, {})
    present = [c for c, v in row.items() if int(v) == 1] if row else []
    manifest.append({"path": p, "volume_name": vname, "findings_present": present,
                     "report": reports.get(vname, reports.get(vname.replace(".nii.gz",""), ""))})
print(f"Manifest: {len(manifest)} volumes. Example findings: {manifest[0]['findings_present'][:4]}")

## 2 · Load Jolia (pinned revision)

In [ ]:
import sys
from transformers import AutoModel
from huggingface_hub import snapshot_download

jolia_repo = snapshot_download("raidium/Jolia", revision=JOLIA_REV)
if jolia_repo not in sys.path:
    sys.path.append(jolia_repo)
from preprocessing_jolia import JoliaPreprocessor

if "jolia" not in globals() or FORCE_RECOMPUTE:
    jolia = AutoModel.from_pretrained(
        "raidium/Jolia", trust_remote_code=True, revision=JOLIA_REV).eval().to(DEVICE)
    pre = JoliaPreprocessor()
    print("Jolia loaded.")
else:
    print("Jolia already loaded in this kernel; skipping.")

ORGAN_NAMES = list(jolia.organ_slot_names)
FOCUS_ORGANS = [o for o in FOCUS_ORGANS if o in ORGAN_NAMES]
ATTN_ORGANS  = [o for o in ATTN_ORGANS if o in ORGAN_NAMES]
assert HERO_ORGAN in ORGAN_NAMES, f"{HERO_ORGAN} not in organ slots"
print("#organ slots:", len(ORGAN_NAMES), "| focus:", FOCUS_ORGANS)

In [ ]:
import numpy as np
import nibabel as nib

def load_hu_volume(path):
    """Load a NIfTI CT as (H,W,D) float32 HU + voxel spacing (mm)."""
    img = nib.load(str(path))
    return img.get_fdata().astype("float32"), tuple(float(z) for z in img.header.get_zooms()[:3])

@torch.no_grad()
def _probe_attention():
    vol, sp = load_hu_volume(manifest[0]["path"])
    x = pre(vol, resolution=sp).unsqueeze(0).to(DEVICE)
    try:
        out = jolia(x, output_organ_queries=True, output_attentions=True)
        for attr in ("organ_attentions", "attentions", "organ_query_attentions"):
            if hasattr(out, attr) and getattr(out, attr) is not None:
                return attr
    except Exception:
        pass
    return None

ATTN_ATTR = _probe_attention()
print("Real attention field:", ATTN_ATTR or "NONE (occlusion fallback)")

## 3 · Embed every volume (cached to `.npz`)

In [ ]:
@torch.no_grad()
def jolia_embed(path):
    vol, sp = load_hu_volume(path)
    x = pre(vol, resolution=sp).unsqueeze(0).to(DEVICE)
    out = jolia(x, output_organ_queries=True)
    return (out.pooler_output.squeeze(0).float().cpu().numpy().astype("float32"),
            out.organ_queries.squeeze(0).float().cpu().numpy().astype("float32"))

n_hit = n_miss = 0
for m in tqdm(manifest, desc="Embeddings"):
    cache = CACHE_ROOT / (m["volume_name"].replace(".nii.gz","") + ".npz")
    if cache.exists() and not FORCE_RECOMPUTE:
        d = np.load(cache); cls, oq = d["cls"], d["organs"]; n_hit += 1
    else:
        cls, oq = jolia_embed(m["path"]); np.savez_compressed(cache, cls=cls, organs=oq); n_miss += 1
    m["cls"] = cls
    organs = {name: oq[i] for i, name in enumerate(ORGAN_NAMES)}
    m["organs_full"] = organs
    m["organs"] = {k: organs[k] for k in FOCUS_ORGANS}
print(f"Embeddings ready. hits: {n_hit}, computed: {n_miss}")

## 4 · Thumbnails (cached)

A 3-slice axial montage per volume. On chest CT the lungs dominate the frame — which is exactly why the
lungs similarity demo will *look* right, not just rank right.

In [ ]:
from PIL import Image

def window(sl, center=40.0, width=400.0):
    lo, hi = center - width/2, center + width/2
    return (np.clip((sl - lo)/(hi - lo), 0, 1) * 255).astype("uint8")

def make_montage(path, out_png):
    vol, _ = load_hu_volume(path)
    D = vol.shape[2]
    tiles = [window(np.rot90(vol[:, :, int(D*f)])) for f in (0.25, 0.50, 0.75)]
    h = max(t.shape[0] for t in tiles)
    tiles = [np.pad(t, ((0, h - t.shape[0]), (0, 0))) for t in tiles]
    Image.fromarray(np.concatenate(tiles, axis=1)).convert("L").save(out_png)
    return out_png

n_made = 0
for m in tqdm(manifest, desc="Thumbnails"):
    out = THUMB_ROOT / (m["volume_name"].replace(".nii.gz","") + ".png")
    if not out.exists() or FORCE_RECOMPUTE:
        make_montage(m["path"], out); n_made += 1
    m["thumb"] = str(out)
print(f"Thumbnails ready ({n_made} created, {len(manifest)-n_made} reused).")

## 5 · Build the FiftyOne dataset (idempotent)

In [ ]:
import fiftyone as fo

need_build = True
if DATASET_NAME in fo.list_datasets() and not FORCE_RECOMPUTE:
    dataset = fo.load_dataset(DATASET_NAME)
    if len(dataset) == len(manifest):
        need_build = False
        print(f"Loaded existing '{DATASET_NAME}' ({len(dataset)}); skipping build.")
    else:
        fo.delete_dataset(DATASET_NAME)

if need_build:
    if DATASET_NAME in fo.list_datasets():
        fo.delete_dataset(DATASET_NAME)
    dataset = fo.Dataset(DATASET_NAME, persistent=True)
    samples = []
    for m in manifest:
        s = fo.Sample(filepath=m["thumb"])
        s["volume_name"]     = m["volume_name"]
        s["nifti_path"]      = str(m["path"])
        s["report"]          = m["report"]
        s["n_findings"]      = len(m["findings_present"])
        s["primary_finding"] = m["findings_present"][0] if m["findings_present"] else "No finding"
        s["has_hero_finding"] = PROBE_FINDING in m["findings_present"]
        if m["findings_present"]:
            s["findings"] = fo.Classifications(
                classifications=[fo.Classification(label=f) for f in m["findings_present"]])
        s["cls_embedding"] = m["cls"].tolist()
        for organ, vec in m["organs"].items():
            s[f"emb_{organ}"] = vec.tolist()
        samples.append(s)
    dataset.add_samples(samples)
    print(f"Built '{DATASET_NAME}' with {len(dataset)} samples.")

print("Distinct primary findings:", dataset.distinct("primary_finding")[:8])

## 6 · Demo 1 — Embedding Atlas → `1 · Embedding Atlas`

UMAP of the global `[CLS]` embedding. In the App: open the **Embeddings** panel *beside* the grid, pick
`jolia_cls_umap`, color by `primary_finding`, and lasso a cluster to filter the grid.

In [ ]:
import fiftyone.brain as fob

def save_view(name, view):
    """Create-or-replace a saved view (idempotent)."""
    if dataset.has_saved_view(name):
        dataset.delete_saved_view(name)
    dataset.save_view(name, view, overwrite=True)
    print("Saved view:", name)

VIZ_KEY = "jolia_cls_umap"
if VIZ_KEY in dataset.list_brain_runs() and not FORCE_RECOMPUTE:
    print(f"Brain run '{VIZ_KEY}' exists; skipping UMAP.")
else:
    if VIZ_KEY in dataset.list_brain_runs():
        dataset.delete_brain_run(VIZ_KEY)
    fob.compute_visualization(dataset, embeddings="cls_embedding", method="umap",
                              brain_key=VIZ_KEY, num_dims=2, seed=51)
    print(f"Computed UMAP -> '{VIZ_KEY}'.")

save_view("1 · Embedding Atlas", dataset.sort_by("n_findings", reverse=True))

## 7 · Demo 2+3 — Per-Organ Similarity → `2 · …(lungs)` and `3 · Organ Contrast (lungs vs liver)`

This is Jolia's party trick: index scans by a **single organ's** embedding — no masks. We build one
similarity index per focus organ, then:

- **`2 · Per-Organ Similarity (lungs)`** — a probe *that actually has a lung finding* followed by its
  **lung**-nearest neighbors. On chest CT the lungs fill the thumbnail, so the shared pathology is visible,
  not just ranked.
- **`3 · Organ Contrast (lungs vs liver)`** — the **same probe**, re-sorted by **liver** similarity. Because
  the liver is barely in a chest scan's field of view, the neighbor set scatters — the visual proof that each
  organ query is a genuinely independent view of the scan.

> **The true liver demo lives on abdominal CT.** To do "find me livers like this one" properly, point this
> notebook at **Merlin-Abd-CT** (abdominal volumes + reports), set `HERO_ORGAN = "liver"` and
> `CONTRAST_ORGAN = "lungs"`, and regenerate. Jolia was trained on chest *and* abdomen, so the liver concept
> query is meaningful there — and the liver actually fills the frame.

In [ ]:
from fiftyone import ViewField as F

# One similarity index per focus organ (guarded per key).
for organ in FOCUS_ORGANS:
    key = f"sim_{organ}"
    if key in dataset.list_brain_runs() and not FORCE_RECOMPUTE:
        continue
    if key in dataset.list_brain_runs():
        dataset.delete_brain_run(key)
    fob.compute_similarity(dataset, embeddings=f"emb_{organ}", brain_key=key)
    print("Indexed", key)

# Pick a probe that HAS the hero finding, so neighbors visibly share pathology.
hero_candidates = dataset.match(F("has_hero_finding") == True)  # noqa: E712
if hero_candidates.count() > 0:
    probe = hero_candidates.first()
    print(f"Probe: {probe.volume_name} (has '{PROBE_FINDING}')")
else:
    probe = dataset.first()
    print(f"No scan with '{PROBE_FINDING}' in this subset; falling back to {probe.volume_name}. "
          f"Consider raising NUM_VOLUMES.")

# View 2: nearest by HERO organ (lungs)
hero_view = dataset.sort_by_similarity(probe.id, brain_key=f"sim_{HERO_ORGAN}", k=8)
save_view(f"2 · Per-Organ Similarity ({HERO_ORGAN})", hero_view)

# View 3: SAME probe, sorted by CONTRAST organ (liver) — neighbors scatter
contrast_view = dataset.sort_by_similarity(probe.id, brain_key=f"sim_{CONTRAST_ORGAN}", k=8)
save_view(f"3 · Organ Contrast ({HERO_ORGAN} vs {CONTRAST_ORGAN})", contrast_view)

print(f"\nSame probe, two organs — watch the neighbors change:")
print(f"  [{HERO_ORGAN}]", [s.primary_finding for s in hero_view][:5])
print(f"  [{CONTRAST_ORGAN}]", [s.primary_finding for s in contrast_view][:5])

## 8 · Demo 4 — Attention Overlays → `4 · Attention Overlays`

Organ cross-attention heatmaps. Uses real attention if the model exposes it (`ATTN_ATTR`); otherwise an
occlusion-saliency fallback on a small `ATTN_GRID`. Overlay PNGs are cached. In the App, toggle
`attn_lungs` (clean, fills the frame) then `attn_pancreas` (smears into neighbors — the paper's honest
Fig. 9 failure mode).

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.cm as cm

@torch.no_grad()
def organ_attention_map(path, organ):
    vol, sp = load_hu_volume(path)
    x = pre(vol, resolution=sp).unsqueeze(0).to(DEVICE)
    slot = ORGAN_NAMES.index(organ)
    if ATTN_ATTR is not None:
        out = jolia(x, output_organ_queries=True, output_attentions=True)
        a = getattr(out, ATTN_ATTR)[0, slot].float().cpu().numpy()
        side = round(len(a) ** (1/3))
        a = a.reshape(side, side, side).mean(axis=2)
        return (a - a.min()) / (a.ptp() + 1e-6)
    base = jolia(x, output_organ_queries=True).organ_queries[0, slot]
    H, W = x.shape[2], x.shape[3]; g = ATTN_GRID
    heat = np.zeros((g, g), "float32"); ys, xs = H // g, W // g
    for gy in range(g):
        for gx in range(g):
            xm = x.clone(); xm[:, :, gy*ys:(gy+1)*ys, gx*xs:(gx+1)*xs, :] = 0
            heat[gy, gx] = float((base - jolia(xm, output_organ_queries=True).organ_queries[0, slot]).norm())
    return (heat - heat.min()) / (heat.ptp() + 1e-6)

def save_overlay(path, organ, out_png):
    vol, _ = load_hu_volume(path)
    base = window(np.rot90(vol[:, :, vol.shape[2] // 2]))
    heat = organ_attention_map(path, organ)
    him = Image.fromarray((heat*255).astype("uint8")).resize(base.shape[::-1], Image.BILINEAR)
    rgba = (cm.jet(np.asarray(him)/255.0)*255).astype("uint8")
    rgb = np.stack([base]*3, -1).astype("uint8")
    Image.fromarray((0.55*rgb + 0.45*rgba[..., :3]).astype("uint8")).save(out_png)
    return out_png

attn_ids = []
# Iterate dataset[:N] for fresh, attached samples. Build/attach ALL organs for a
# sample, then s.save() ONCE outside the organ loop — a single atomic write avoids
# the partial-commit window that can drop an organ's heatmap between iterations.
for s in tqdm(dataset[:min(ATTN_SAMPLES, len(dataset))], desc="Attention overlays"):
    attn_ids.append(s.id)
    for organ in ATTN_ORGANS:
        out = OVERLAY_ROOT / f"{s.volume_name.replace('.nii.gz','')}_{organ}.png"
        if not out.exists() or FORCE_RECOMPUTE:
            save_overlay(Path(s.nifti_path), organ, out)
        assert out.exists(), f"overlay PNG not written for {organ}: {out}"
        s[f"attn_{organ}"] = fo.Heatmap(map_path=str(out))
    s.save()  # commit both heatmaps together

save_view("4 · Attention Overlays", dataset.select(attn_ids))
print("Overlays attached to", len(attn_ids), "scans:",
      [f for f in dataset.get_field_schema() if f.startswith("attn_")],
      "| Source:", ATTN_ATTR or "occlusion fallback")

## 9 · Demo 5 — Findings Probe → `5 · Probe Errors (FP+FN)`

Logistic-regression probe on the frozen `[CLS] + lungs` feature for a balanced finding; predictions written
back, disagreements saved for inspection alongside `report`. Demonstrates the workflow, not the paper's
AUROC (too few volumes).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from collections import Counter

organ = HERO_ORGAN
X = np.array([m["cls"].tolist() + m["organs"][organ].tolist() for m in manifest], "float32")

counts = Counter(f for m in manifest for f in m["findings_present"])
target = next((f for f, c in counts.most_common()
               if len(manifest)*0.25 <= c <= len(manifest)*0.75), None) \
         or (counts.most_common(1)[0][0] if counts else None)

if target is None:
    print("No findings in subset — increase NUM_VOLUMES.")
else:
    y = np.array([int(target in m["findings_present"]) for m in manifest])
    print(f"Probing '{target}' (positives {y.sum()}/{len(y)}) on [CLS]+{organ}")
    idx = np.arange(len(y))
    tr, _ = train_test_split(idx, test_size=0.4, random_state=51,
                             stratify=y if y.sum() > 1 else None)
    clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X[tr], y[tr])
    proba = clf.predict_proba(X)[:, 1]

    id_by_name = {s.volume_name: s.id for s in dataset}
    for m, p in zip(manifest, proba):
        s = dataset[id_by_name[m["volume_name"]]]
        gt, pred = int(target in m["findings_present"]), int(p >= 0.5)
        s["probe_target"] = target
        s["probe_score"]  = float(p)
        s["probe_pred"]   = "positive" if pred else "negative"
        s["probe_gt"]     = "positive" if gt else "negative"
        s["probe_error"]  = "TP" if (gt and pred) else "TN" if (not gt and not pred) \
                            else "FP" if pred else "FN"
        s.save()

    errs = dataset.match(F("probe_error").is_in(["FP", "FN"]))
    save_view("5 · Probe Errors (FP+FN)", errs)
    print(f"Wrote probe fields. FP+FN count: {errs.count()}")

## 10 · Launch the app

Pick each demo from the **view selector** dropdown. Suggested tour:

1. **`1 · Embedding Atlas`** — Embeddings panel beside the grid → `jolia_cls_umap` → color by
   `primary_finding`, lasso a cluster.
2. **`2 · Per-Organ Similarity (lungs)`** — probe + its lung-nearest neighbors; they share lung pathology,
   and the lungs fill the montage so you can *see* it.
3. **`3 · Organ Contrast (lungs vs liver)`** — the **same probe**, now sorted by liver. Say it out loud:
   *"Same scan, different organ, different neighbors."* The scatter is the point.
4. **`4 · Attention Overlays`** — toggle `attn_lungs` (clean) then `attn_pancreas` (smears — honest failure).
5. **`5 · Probe Errors (FP+FN)`** — read `report` on each miss: model error or noisy label?

In [ ]:
print("Saved views:", dataset.list_saved_views())
session = fo.launch_app(dataset)
session

---
### Re-running, scaling & the abdominal liver demo

- **Cached & idempotent.** Re-run top-to-bottom anytime; only new work runs. Grow `NUM_VOLUMES` to pull &
  embed more scans incrementally. Pass `create_index=True` to `compute_visualization` for snappy lassoing at
  scale. Force a clean rebuild with `FORCE_RECOMPUTE = True` or `fo.delete_dataset(DATASET_NAME)`.
- **The real liver demo → abdomen.** Swap the §1 download to **Merlin-Abd-CT**, set `HERO_ORGAN="liver"`,
  `CONTRAST_ORGAN="lungs"`, `PROBE_FINDING` to an abdominal finding (e.g. `"Liver lesion"`), and
  `ATTN_ORGANS=["liver","pancreas"]`. The liver then fills the frame and the demo is both correct and
  legible.
- **Real attention.** If `ATTN_ATTR` printed a field name, overlays already use true concept-query attention.

*Model: Raidium Jolia (research preview, not for clinical use). Data: CT-RATE (gated, research license).*